# Speed Test from USDF S3 to IDF

## Imports

In [ ]:
import asyncio
import datetime
import json
import subprocess
import threading
import statistics
import uuid

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Self, override

import lsst.rsp

## Parameters and globals

In [ ]:
# You'll have to regenerate a URL each week; make sure you pick something without sensitive data.
# This URL is in a public notebook and is by design world-readable.
URL='https://sdfdatas3.slac.stanford.edu/rubin-dp02-products/u/ctslater/DM-33345-tract4431/20220125T013610Z/matchedCatalogTract/4431/u/matchedCatalogTract_LSSTCam-imSim_4431_u_DC2_u_ctslater_DM-33345-tract4431_20220125T013610Z.fits?AWSAccessKeyId=dp02user&Signature=nVN1dvV4h%2BRqnE6b53cmGLlWLOs%3D&Expires=1774896253'
FILESIZE=1334496960
# Or some IPAC data, either from S3 at AWS or from Caltech
#URL='https://nasa-irsa-euclid-q1.s3.amazonaws.com/q1/MER/102158585/VIS/EUC_MER_BGSUB-MOSAIC-VIS_TILE102158585-B59B9_20241025T003538.515692Z_00.00.fits'
#URL='https://irsa.ipac.caltech.edu/ibe/data/euclid/q1/MER/102158585/VIS/EUC_MER_BGSUB-MOSAIC-VIS_TILE102158585-B59B9_20241025T003538.515692Z_00.00.fits'
#FILESIZE=1474565760


# CURL_CONCURRENCY defaults to 50 and doesn't really help; the -Z option parallelizes across multiple URLs.
# Still, leave it in here, because we may want to adapt this to pulling many different files concurrently.
CURL_CONCURRENCY=100
CMD="curl"

# Change to suit.
OUTPUT_DIR="/rubin/shared/speedtest"

# We have some shared data structures which we protect with a global lock.
lock=threading.Lock()

## Class and functions for executing a speedtest

In [ ]:
@dataclass
class Results:
    """Hold results of a curl speedtest."""
    mean: float
    median: float
    aggregate: float
    effective: float
    real: float | None = None
    node: str | None = None
    start: float | None = None
    concurrency: int | None = None
    end: float | None = None
    
    @classmethod
    def from_list(cls, inp: list[float]) -> Self:
        """The list will contain the stuff curl can tell us.
        The other fields have to be filled in from the
        caller.
        
        The most useful field is `real`, because that's
        the actual wallclock time, and what we really care about
        is how long it takes to get the bytes from USDF to IDF.
        """
        mean = statistics.mean(inp)
        median = statistics.median(inp)
        aggregate = sum(inp)
        effective = len(inp) * min(inp)
        return cls(
            mean=mean,
            median=median,
            aggregate=aggregate,
            effective=effective,
            real=None,
            node=None,
            concurrency=None,
            start=None,
            end=None,
        )

    @override
    def __str__(self) -> str:
        """Pretty-print our results."""
        ret = (f"mean: {self.mean:02.2f}MBps"
               f" median: {self.median:02.2f}MBps"
               f" aggregate {self.aggregate:02.2f}MBps"
               f" effective {self.effective:02.2f}MBps"
              )
        if self.real:
            ret += f" real {self.real:02.2f}MBps"
        if self.node:
            ret += f" node {self.node}"
        if self.concurrency:
            ret += f" concurrency {self.concurrency}"
        if self.start:
            ret += f" start {datetime.datetime.fromtimestamp(self.start, tz=datetime.UTC)}"
        if self.end:
            ret += f" end {datetime.datetime.fromtimestamp(self.end, tz=datetime.UTC)}"
        return ret
        
"""Shared globals, which we need the lock to protect."""
multiprocess_results: dict[int, Results] = {}
results: list[float] = []

async def run_curl(cmd_args: list[str], results: list[float]) -> None:
    """Single curl invocation.  This function runs in a coroutine, and so does the curl subprocess.
    """
    proc = await asyncio.create_subprocess_exec(CMD, *cmd_args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    await proc.wait()  # This, of course, blocks while curl downloads.
    out = (await proc.stdout.read()).decode()
    err = (await proc.stderr.read()).decode()
    # Curl speed report goes to stderr.  (stdout would be for the download data, but
    # we throw it away with `-o /dev/null`)
    # Last line of results is completed download.  Last field is download speed.    
    speed = err.split("\r")[-1].rstrip().split()[-1]
    # Last character of that is "M", for MBps, or "k", for Kbps
    # curl uses binary rather than decimal units
    if speed.endswith("k"):
        # convert it to MBps
        print(f"Converting speed in {speed} k to M")
        f_speed=(float((err.split("\r")[-1].rstrip().split()[-1])[:-1]))
        f_speed = f_speed / 2**10  # Convert to MBbps; 1024, not 1000
        lock.acquire()
        results.append(f_speed)
        lock.release()
    elif speed.endswith("M"):
        # Our preferred units are MBps
        f_speed=float((err.split("\r")[-1].rstrip().split()[-1])[:-1])
        print(f"speed: {f_speed}MBps")
        lock.acquire()
        results.append(f_speed)
        lock.release()
    else:
        # We might someday have to add more suffixes; for now, if it's
        # not M or k we don't know what to do.
        print(f"Uninterpretable speed {speed}")
    return
    
async def spawn_curls(concurrency: int, cmd_args: list[str]) -> Results:
    """Launch multiple curl subprocesses concurrently in a task group."""
    results: list[float] = []
    async with asyncio.TaskGroup() as tg:
        tasks: set[asyncio.Task] = set()
        then=datetime.datetime.now(tz=datetime.UTC)
        for task in range(concurrency):
            tasks.add(tg.create_task(run_curl(cmd_args, results)))

    now=datetime.datetime.now(tz=datetime.UTC)
    node=lsst.rsp.get_node()
    res=Results.from_list(results)
    res.node=node
    realspeed=((concurrency*FILESIZE) / (now - then).total_seconds()) / 2**20
    res.real = realspeed
    res.start = then.timestamp()
    res.end = now.timestamp()
    res.concurrency=concurrency
    return res

async def run_test(concurrency: int) -> None:
    """Run a test at a given concurrent level of curl processes, and store the results."""
    # With only one target URL the `--parallel` options don't do anything.
    cmd_args=[
        "--output",
        "/dev/null",
        "--parallel",
        "--parallel-max",
        str(CURL_CONCURRENCY),
        "--parallel-immediate",
        "--url",
        URL
    ]
    results = []  # Reset for each run
    pl="" if concurrency < 2 else "s in parallel"
    print(f"{concurrency} curl{pl}")
    then=datetime.datetime.now(tz=datetime.UTC)
    p_results = await spawn_curls(concurrency, cmd_args)   
    lock.acquire()
    multiprocess_results[concurrency] = p_results
    lock.release()

async def multi_test() -> None:
    """Run each concurrency test sequentially."""
    for concurrency in [ 1, 3, 10, 30]:
        # 100 doesn't complete on a 4CPU 16GB Pod
        # So we stop at 30.
        await run_test(concurrency)

## Run the test and generate the output

In [ ]:
# Run the test
await multi_test()

# Write output
o_dir = Path(OUTPUT_DIR)
if not o_dir.is_dir():
    o_dir.mkdir()
    o_dir.chmod(0o777)  # Yeah, world-writeable.
for k,v in multiprocess_results.items():
    f_uuid=uuid.uuid4()
    # Write to a random filename; we're gonna suck it all up with glob() anyway
    fname = f"{f_uuid}-results.json"
    o_file = o_dir / fname
    o_file.write_text(json.dumps(asdict(v)))
    print(v)